In [1]:
import os
import cv2
import numpy as np
from pathlib import Path

# ---------------- CONFIG ----------------
ROOT = Path(
    "SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions"
) / "SnowPole_Detection_Dataset"

OUT_ROOT = Path("dataset_fused_anisotropic/images")

MODALITIES = ["reflec", "signal", "nearir", "range"]
SPLITS = ["train", "valid", "test"]
IMG_EXTS = [".png", ".jpg", ".jpeg"]

NUM_ITER = 8
KAPPA = 25
GAMMA = 0.15
# ----------------------------------------


def anisotropic_diffusion(img, num_iter=10, kappa=30, gamma=0.1):
    img = img.astype(np.float32)

    for _ in range(num_iter):
        north = np.roll(img, -1, axis=0) - img
        south = np.roll(img, 1, axis=0) - img
        east  = np.roll(img, -1, axis=1) - img
        west  = np.roll(img, 1, axis=1) - img

        c_n = np.exp(-(north / kappa)**2)
        c_s = np.exp(-(south / kappa)**2)
        c_e = np.exp(-(east  / kappa)**2)
        c_w = np.exp(-(west  / kappa)**2)

        img = img + gamma * (
            c_n * north +
            c_s * south +
            c_e * east +
            c_w * west
        )

    return img


def read_first_channel(path):
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(path)

    if img.ndim == 3:
        img = img[:, :, 0]

    return img.astype(np.float32)


def normalize(img):
    img = img.astype(np.float32)
    mean = img.mean()
    std = img.std() + 1e-6
    return (img - mean) / std


In [2]:
for split in SPLITS:
    print(f"Processing split: {split}")

    out_dir = OUT_ROOT / split
    out_dir.mkdir(parents=True, exist_ok=True)

    refl_dir = ROOT / MODALITIES[0] / split
    images = [p for p in refl_dir.iterdir() if p.suffix.lower() in IMG_EXTS]

    for img_path in images:
        imgs = []

        for mod in MODALITIES:
            mod_path = ROOT / mod / split / img_path.name
            if not mod_path.exists():
                raise FileNotFoundError(f"Missing: {mod_path}")

            img = read_first_channel(mod_path)
            img = normalize(img)

            img = anisotropic_diffusion(
                img,
                num_iter=NUM_ITER,
                kappa=KAPPA,
                gamma=GAMMA
            )

            imgs.append(img)

        # Equal-weight fusion
        fused = np.mean(imgs, axis=0)

        # Rescale to uint8
        fused = fused - fused.min()
        fused = fused / (fused.max() + 1e-6)
        fused = (fused * 255).astype(np.uint8)

        cv2.imwrite(str(out_dir / img_path.name), fused)

    print(f"Finished split: {split}")

print("Anisotropic fusion dataset created successfully.")

Processing split: train
Finished split: train
Processing split: valid
Finished split: valid
Processing split: test
Finished split: test
Anisotropic fusion dataset created successfully.


In [3]:
from ultralytics import YOLO

print("Starting YOLO training with 4-channel input...")

model = YOLO("yolo11n.yaml")

model.train(
    data="weighted_fusion.yaml",
    imgsz=1024,
    epochs=500,
    patience=40,        # early stopping
    batch=8,
    device=0,
    project="weighted_late_gate_experiments",
    name="weighted_yolo11n_late_gate",
    amp=False,
    augment=False,
    workers=0,
)

Starting YOLO training with 4-channel input...
New https://pypi.org/project/ultralytics/8.4.14 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.6  Python-3.11.0 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=weighted_fusion.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, mo

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000274A823E910>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480